# SMART — SAM copy-paste production release

This notebook creates a **train-only** augmentation release. Validation is never copied, linked, or modified. The same generated delta is later linked to the raw base for Mejora B and to the LaMa base for Mejora C.

In [ ]:
import os, subprocess, sys
from pathlib import Path

RUN_ID = 'sam_cp_production_v1'  # Change this only for a deliberate new release.
RESUME = False  # Set True only after an interrupted run with the same RUN_ID.
BUDGET_FRACTION = 1.0  # Policy already caps every augmented class at 50% synthetic.
ROOT = Path('/kaggle/working/augmentation_production')
REPO = ROOT / 'ia_article'
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used,utilization.gpu', '--format=csv,noheader'], check=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'chore/augmentation-evidence', '--filter=blob:none', '--sparse', 'https://github.com/unsa-semester-2026-A/ia_article.git', str(REPO)], check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'experiments'], cwd=REPO, check=True)
os.chdir(REPO / 'experiments')
# Preserve Kaggle's CUDA Torch; install only the project and pinned Ultralytics code.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'ultralytics==8.4.103'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
from ultralytics import SAM
print('Ultralytics SAM import: OK')
subprocess.run([sys.executable, '-m', 'pytest', '-o', 'addopts=', 'src/augmentation', '-q'], check=True)


In [ ]:
input_root = Path('/kaggle/input')
dataset_root = next((p for p in (input_root / 'mtc-challenge', input_root / 'datasets/alvaroquispeunsa/mtc-challenge') if p.is_dir()), None)
if dataset_root is None: raise FileNotFoundError(f'MTC dataset not attached; input={[p.name for p in input_root.iterdir()]}')
STATIC = dataset_root / 'static_vehicles.json'
LABELS_ROOT = dataset_root / 'yolo_obb_labels'
LABELS = LABELS_ROOT / 'train' if (LABELS_ROOT / 'train').is_dir() else LABELS_ROOT
RAW = dataset_root / 'train_resized' / 'train'
LAMA = dataset_root / 'smart_lama_corrected' / 'train'
METADATA = dataset_root / 'split_metadata.csv'
for path in (STATIC, LABELS, RAW, LAMA, METADATA):
    if not path.exists(): raise FileNotFoundError(path)
print({'dataset_root': str(dataset_root), 'labels': len(list(LABELS.glob('*.txt'))), 'raw': RAW.is_dir(), 'lama': LAMA.is_dir()})


In [ ]:
cmd = [sys.executable, '-m', 'src.augmentation.run', 'production',
    '--split-metadata', str(METADATA), '--static-vehicles', str(STATIC),
    '--labels-train', str(LABELS), '--raw-images', str(RAW), '--lama-images', str(LAMA),
    '--output-dir', str(ROOT / 'outputs'), '--run-id', RUN_ID,
    '--budget-fraction', str(BUDGET_FRACTION), '--no-drive-sync']
if RESUME: cmd.append('--resume')
subprocess.run(cmd, check=True)
OUTPUT = ROOT / 'outputs' / f'sam_copy_paste_{RUN_ID}'
RELEASE = OUTPUT / 'release' / f'sam_copy_paste_{RUN_ID}'
assert list((RELEASE / 'images' / 'train').glob('*.jpg'))
assert list((RELEASE / 'labels' / 'train').glob('*.txt'))
assert not (RELEASE / 'images' / 'val').exists()
assert not (RELEASE / 'labels' / 'val').exists()
for name in (f'sam_copy_paste_delta_{RUN_ID}.zip', f'sam_copy_paste_audit_{RUN_ID}.zip', 'production_summary.json'):
    assert (OUTPUT / name).is_file(), name
print((OUTPUT / 'production_summary.json').read_text())
print('Kaggle dataset upload folder:', RELEASE)
